# InSpatio-WorldFM Demo

**Requirements:** GPU runtime — Runtime → Change runtime type → T4 GPU (or better)

**How to run:** This notebook has two parts separated by an automatic runtime restart.
1. Run **Part 1** (cells below) — installs conda, then the runtime restarts automatically
2. After the restart, run **Part 2** to set up the environment and run the demo

## Part 1 — Install conda
Run this cell. The runtime will restart automatically — then continue with Part 2.

In [ ]:
!nvidia-smi
!pip install -q condacolab
import condacolab
condacolab.install()  # triggers automatic runtime restart

## Part 2 — Clone repo, set up environment, run demo
Start here after the runtime restarts.

In [ ]:
# Clone your fork and initialize submodules
import os
!git clone https://github.com/joanamizrahi-png/worldfm.git
os.chdir('worldfm')
!git submodule update --init --recursive

In [ ]:
# Create conda environment with Python 3.10 (matching what WorldFM was built for)
!conda create -n WorldFM python=3.10 -y -q

# Install PyTorch 2.5 with CUDA 12.4
!conda run -n WorldFM pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 \
    --index-url https://download.pytorch.org/whl/cu124 -q

In [ ]:
# Install all requirements into the conda env
!conda run -n WorldFM pip install -r requirements.txt --ignore-requires-python -q 2>&1 | tail -20

# Install packages that need explicit handling
!conda run -n WorldFM pip install mmengine==0.10.7 -q
!conda run -n WorldFM pip install \
    git+https://github.com/EasternJournalist/utils3d.git@c5daf6f6c244d251f252102d09e9b7bcef791a38 -q

# mmcv must match PyTorch/CUDA version
!conda run -n WorldFM pip install mmcv==1.7.2 \
    -f https://download.openmmlab.com/mmcv/dist/cu124/torch2.5/index.html -q

In [ ]:
# Build submodule packages
!conda run -n WorldFM bash -c \
    'cd submodules/Real-ESRGAN && pip install basicsr-fixed facexlib gfpgan -q && python setup.py develop -q'
!conda run -n WorldFM bash -c 'cd submodules/ZIM && pip install -e . -q'

In [ ]:
# Download pretrained model weights (VAE + DMD, several GB, takes a few minutes)
!conda run -n WorldFM python download_ckpts.py

In [ ]:
# Run demo on the included mario.png scene
!conda run -n WorldFM python run_pipeline.py \
    --meta demo/meta.json --output_dir outputs --save_mode image

In [ ]:
# View generated images
import glob
from IPython.display import Image, display

output_files = sorted(glob.glob('outputs/**/*.png', recursive=True))
print(f'Generated {len(output_files)} images')
for f in output_files[:10]:
    display(Image(f))